In [1]:
#1. Agri-Tech — Smart Farming Sensor Prediction

#Context: Smart Farming Sensors|Sensor records: 1000
#Inputs: Soil_Moisture (%); UV_Index (0-15); Pesticide_Residue (ppm)
#Target used for the runnable regression example: Crop_Health_Score — predicted crop health score (0–100).

# ANN Level 1 - FULL PROGRAM (Worksheet + Answers combined)
# 1. Agri-Tech — Smart Farming Sensor Prediction (Regression)

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
torch.manual_seed(42)
np.random.seed(42)
print("Libraries imported successfully! PyTorch Version:", torch.__version__)

Libraries imported successfully! PyTorch Version: 2.13.0+cpu


In [6]:
# [CELL 2] Data Pipeline, Normalization & Tensor Conversion

if not os.path.exists('agri_tech_sensors.csv'):
    n = 1000
    demo_df = pd.DataFrame({
        'Soil_Moisture': np.random.uniform(10, 90, n),
        'UV_Index': np.random.uniform(0, 15, n),
        'Pesticide_Residue': np.random.uniform(0, 5, n),
    })
  
    demo_df['Crop_Health_Score'] = 20 + 0.75*demo_df['Soil_Moisture'] - 1.5*demo_df['UV_Index'] - 8*demo_df[
'Pesticide_Residue'] + np.random.normal(0, 4, n)
    for col in ['Soil_Moisture', 'UV_Index']:
        mask = np.random.rand(n) < 0.03
        demo_df.loc[mask, col] = np.nan
    demo_df.to_csv('agri_tech_sensors.csv', index=False)
df = pd.read_csv('agri_tech_sensors.csv')
feature_cols = ['Soil_Moisture', 'UV_Index', 'Pesticide_Residue']
X = df[feature_cols].values
y = df['Crop_Health_Score'].values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
print(f"X_train shape: {X_train_tensor.shape}, y_train shape: {y_train_tensor.shape}")

X_train shape: torch.Size([800, 3]), y_train shape: torch.Size([800, 1])


In [5]:
# [CELL 3] Designing the ANN Architecture for Regression

class AgriTechRegressor(nn.Module):
    def __init__(self):
        super(AgriTechRegressor, self).__init__()   
        self.linear1 = nn.Linear(3, 32)  
        self.linear2 = nn.Linear(32, 16)
        self.linear3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x
model = AgriTechRegressor()
print(model)

AgriTechRegressor(
  (linear1): Linear(in_features=3, out_features=32, bias=True)
  (linear2): Linear(in_features=32, out_features=16, bias=True)
  (linear3): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
)


In [7]:
# [CELL 4] Loss Function & Optimizer

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Loss function: MSELoss | Optimizer: Adam")

Loss function: MSELoss | Optimizer: Adam


In [9]:
# [CELL 5] The Training Loop

num_epochs = 100
batch_size = 32
print("Training started...\n")
for epoch in range(num_epochs):
    for i in range(0, len(X_train_tensor), batch_size):
        X_batch = X_train_tensor[i:i+batch_size]
        y_batch = y_train_tensor[i:i+batch_size]
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], MSE Loss: {loss.item():.4f}")
print("\nTraining complete!")

Training started...

Epoch [20/100], MSE Loss: 32.3349
Epoch [40/100], MSE Loss: 18.4305
Epoch [60/100], MSE Loss: 17.6010
Epoch [80/100], MSE Loss: 17.5657
Epoch [100/100], MSE Loss: 17.5626

Training complete!


In [10]:
# [CELL 6] The Evaluation Loop

model.eval()
with torch.no_grad():
    test_predictions = model(X_test_tensor)
test_mse = loss_fn(test_predictions, y_test_tensor).item()
test_rmse = np.sqrt(test_mse)
print("Test Set Evaluation:")
print(f" - Mean Squared Error (MSE) : {test_mse:.4f}")
print(f" - Root Mean Squared Error (RMSE): {test_rmse:.4f}\n")
print(f"{'Actual':<15} {'Predicted':<15} {'Abs Error':<15}")
print("-" * 45)
with torch.no_grad():
    for i in range(5):
        actual = y_test_tensor[i].item()
        predicted = test_predictions[i].item()
        error = abs(actual - predicted)
        print(f"{actual:<15.2f} {predicted:<15.2f} {error:<15.2f}")

Test Set Evaluation:
 - Mean Squared Error (MSE) : 26.0804
 - Root Mean Squared Error (RMSE): 5.1069

Actual          Predicted       Abs Error      
---------------------------------------------
25.78           23.92           1.86           
55.21           51.47           3.74           
14.74           22.16           7.42           
16.09           16.93           0.85           
73.15           76.55           3.40           


In [11]:
# [CELL 7] Real-World Inference

raw_input = np.array([[62.0, 6.5, 1.2]])
imputed_data = imputer.transform(raw_input)
scaled_data = scaler.transform(imputed_data)
input_tensor = torch.tensor(scaled_data, dtype=torch.float32)
model.eval()
with torch.no_grad():
    predicted_score = model(input_tensor).item()
print(f"Predicted Crop Health Score: {predicted_score:.2f} / 100")


Predicted Crop Health Score: 47.60 / 100
